        # 🔁 L07　交叉驗證與隨機森林
        **統計冒險之旅 2026**　｜　Day 3（10/05 一）🗻 預測之巔　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch5、Ch8；資料：勇者咖啡會員


        ### 🎯 這一關你會學到
        - k-fold 交叉驗證、三份切分、Pipeline 防資料洩漏
- bootstrap → bagging → 隨機森林
- 決策樹直覺與特徵重要性

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L07"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["7-1", "7-2", "7-3", "7-4", "7-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_7_1(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "X"), 列=2000, 欄=10)
    if not ok: return (False, msg)
    if not 約等於(抓變數(ns, "回購率"), 0.48400, 0.001): return (False, "回購率 = y.mean()。")
    if int(抓變數(ns, "特徵數")) != 12: return (False, "特徵數是訓練集經 one-hot 後的欄位數。")
    if len(抓變數(ns, "X_train")) != 1500 or len(抓變數(ns, "X_test")) != 500: return (False, "先用指定參數保留 25% final test：訓練 1500、測試 500 筆。")
    return (len(抓變數(ns, "y_train")) == 1500 and len(抓變數(ns, "y_test")) == 500, "X、y 的切分列數要一致。")
任務定義("7-1", _check_7_1, 提示="先切 raw X/y，再用 ColumnTransformer 包住 OneHotEncoder；類別只能從訓練資料學。")

def _check_7_2(run):
    out, ns = run()
    if len(抓變數(ns, "每摺AUC")) != 5: return (False, "cv=5 會有 5 個分數。")
    if not 約等於(抓變數(ns, "平均AUC"), 0.82335, 0.005): return (False, "平均AUC 不對：交叉驗證只能使用 X_train、y_train，scoring='roc_auc'，Pipeline 含 StandardScaler。")
    return (約等於(抓變數(ns, "AUC標準差"), 0.01687, 0.005), "AUC標準差 = 每摺AUC.std()；final test 此時仍不可使用。")
任務定義("7-2", _check_7_2, 提示="cross_val_score 回傳一個陣列，.mean() 與 .std()。")

def _check_7_3(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "OOF_AUC"), 0.82128, 0.005): return (False, "OOF_AUC 要用訓練資料的 out-of-fold 預測計算，不能先看 final test。")
    if len(抓變數(ns, "AUC們")) != 500: return (False, "請對訓練資料的 OOF 預測 bootstrap 500 次。")
    return (約等於(抓變數(ns, "AUC下界"), 0.8018, 0.01) and 約等於(抓變數(ns, "AUC上界"), 0.8400, 0.01), "np.percentile(AUC們, 2.5) 與 97.5；資料應為 y_train 與 OOF機率。")
任務定義("7-3", _check_7_3, 提示="cross_val_predict(..., X_train, y_train, method='predict_proba') 取得 OOF 機率，再在訓練資料內 bootstrap。")

def _check_7_4(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "樹CV平均"), 0.65130, 0.02): return (False, "樹CV平均要在 X_train、y_train 上做 5 摺 CV；不要用 final test 選模型。")
    if not 約等於(抓變數(ns, "森林CV平均"), 0.79180, 0.02): return (False, "森林CV平均不對：訓練內 5 摺 CV，n_estimators=300, random_state=42。")
    if str(抓變數(ns, "選定模型")) != "隨機森林": return (False, "依 CV 平均 AUC 選擇模型；本資料應選隨機森林。")
    if not 約等於(抓變數(ns, "最終測試AUC"), 0.79023, 0.01): return (False, "完成選模後才 fit 最終模型，final test 只用這一次報告 AUC。")
    return (bool(抓變數(ns, "樹在背考古題")) == True, "比較樹的訓練 AUC 與訓練內 CV AUC，可看出長到底的樹正在背題。")
任務定義("7-4", _check_7_4, 提示="先用 X_train、y_train 的 CV 比較，再選模型；最後才用 X_test、y_test 報告一次。")

def _check_7_5(run):
    out, ns = run()
    s = 抓變數(ns, "重要性")
    ok, msg = 資料框像(s, 列=12, 種類="Series")
    if not ok: return (False, msg)
    if s.iloc[0] < s.iloc[-1]: return (False, "要由大到小排序（ascending=False）。")
    if str(抓變數(ns, "最重要")) != "距上次來店天數": return (False, "最重要 = 重要性.index[0]。")
    return (any("重要" in f["title"] for f in run.figs), "圖的標題要包含「重要」。")
任務定義("7-5", _check_7_5, 提示="sort_values(ascending=False)。")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
members = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0-rc.1/data/coffee_members.csv")      # 勇者咖啡 2,000 位會員，回購 = 30 天內有沒有再來
members.head()

## 🔁 7-1　先封存 final test，再讓訓練資料輪流當模擬考
Day 2 我們切一次訓練／測試就評分。但若反覆看測試分數再改模型，測試集也會變成考古題，分數不再公正。
**k 摺交叉驗證（k-fold cross-validation）**：先封存 final test，只把訓練資料切成 k 份，輪流讓其中一份當驗證、其他份用來訓練。跑 k 次取平均，也觀察標準差。
`cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")` 一行完成訓練內 CV。

> 🧭 三份角色：訓練資料用來估參數，訓練內 CV（或獨立 validation/development）用來比較與選模型，final test 只在所有選擇完成後開封一次。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
X = members.drop(columns=["會員編號", "回購"])
y = members["回購"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
類別欄 = ["性別", "會員等級", "最愛類別"]
數值欄 = [c for c in X.columns if c not in 類別欄]
preprocess = ColumnTransformer([("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), 類別欄), ("num", "passthrough", 數值欄)], verbose_feature_names_out=False)
pipe = Pipeline([("preprocess", preprocess), ("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
每摺 = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
print("訓練內 5 摺 AUC：", 每摺.round(3), "→ 平均", round(每摺.mean(), 3), "± 標準差", round(每摺.std(), 3))
print("final test 已封存：", len(X_test), "筆；選模完成前不查看分數")

## 7-2　資料洩漏：考前偷看考卷
最常見的錯：先用**全部資料**探索 one-hot 類別、標準化或選特徵，再切分——測試集的資訊已經偷偷流進訓練。**Pipeline** 把「one-hot → 標準化 → 模型」包在一起，交叉驗證時每一摺都只用該摺的訓練部分 fit 前處理。
這一課的規矩：先 raw split，再把 `ColumnTransformer(OneHotEncoder)` 與模型一起放入 `Pipeline`。

## 7-3　OOF ＋ bootstrap：先在訓練資料看不確定性
Day 1 用 bootstrap 估平均的區間；這裡先用訓練資料的 out-of-fold（OOF）預測取得未參與該筆訓練的機率，再對 OOF 結果抽後放回 500 次估 AUC 區間。final test 仍保持封存。

In [ ]:
OOF機率 = cross_val_predict(pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
OOF_AUC = roc_auc_score(y_train, OOF機率)
rng = np.random.default_rng(42)
索引 = np.arange(len(y_train)); yv = y_train.values
AUC們 = []
for _ in range(500):
    s = rng.choice(索引, len(索引), replace=True)
    if np.unique(yv[s]).size < 2: continue
    AUC們.append(roc_auc_score(yv[s], OOF機率[s]))
print("訓練內 OOF AUC：", round(OOF_AUC, 3), "；bootstrap 95% 區間：", np.percentile(AUC們, [2.5, 97.5]).round(3))

## 7-4　從一棵樹到一座森林
**決策樹**像玩 20 個問題：「距上次來店超過 30 天嗎？→ 是 → 來店次數少於 5 嗎？→ …」一路問到底。好懂，但一棵長到底的樹很可能把訓練會員背起來（訓練 AUC = 1.0，訓練內 CV 明顯較低）——高變異。
**隨機森林**：用 bootstrap 抽很多份資料、各長一棵樹，每棵樹只看部分特徵，最後投票。樹與森林要先用相同的訓練內 CV 比較；選定後，才開封 final test 報告一次。

In [ ]:
tree = Pipeline([("preprocess", preprocess), ("model", DecisionTreeClassifier(random_state=42))])
rf = Pipeline([("preprocess", preprocess), ("model", RandomForestClassifier(n_estimators=300, random_state=42))])
樹CV = cross_val_score(tree, X_train, y_train, cv=5, scoring="roc_auc")
森林CV = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
print("訓練內 CV｜一棵樹", round(樹CV.mean(), 3), "｜隨機森林", round(森林CV.mean(), 3))
print("依 CV 暫選：", "隨機森林" if 森林CV.mean() >= 樹CV.mean() else "決策樹", "；final test 繼續封存")

### 🎯 任務 7-1　資料準備

建立 raw `X`（只去掉 會員編號、回購）與 `y`（回購），先用 `test_size=0.25, random_state=42, stratify=y` 切出 `X_train, X_test, y_train, y_test`。再定義 `類別欄`、`數值欄` 與 `preprocess`（`ColumnTransformer` 包含 `OneHotEncoder(drop='first', handle_unknown='ignore')`）；類別只能由訓練資料學得。算 `回購率`，並將訓練集 fit 後的轉換欄位數存成 `特徵數`；final test 從現在起封存。

In [ ]:
# 🎯 任務 7-1　資料準備（請保留這一行）
X = members.drop(columns=["會員編號", "回購"])
y = members["回購"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
類別欄 = ["性別", "會員等級", "最愛類別"]
數值欄 = [c for c in X.columns if c not in 類別欄]
preprocess = ColumnTransformer([("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), 類別欄), ("num", "passthrough", 數值欄)], verbose_feature_names_out=False)
回購率 = ???
特徵數 = len(preprocess.fit(X_train).get_feature_names_out())
print(preprocess.get_feature_names_out().tolist()); print(round(回購率, 3), 特徵數, len(X_train), len(X_test))

In [ ]:
檢查("7-1")   # ◀ 執行這一格，看看任務 7-1 有沒有過關

### 🎯 任務 7-2　五摺交叉驗證

只使用 raw `X_train, y_train`，以 Pipeline（ColumnTransformer/OneHotEncoder → StandardScaler → LogisticRegression(max_iter=1000)）做 5 摺交叉驗證（`scoring='roc_auc'`），存成 `每摺AUC`，算 `平均AUC` 與 `AUC標準差`。每摺都要自己 fit one-hot，不要查看 final test。

In [ ]:
# 🎯 任務 7-2　五摺交叉驗證（請保留這一行）
pipe = Pipeline([("preprocess", preprocess), ("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
每摺AUC = cross_val_score(pipe, X_train, y_train, cv=???, scoring=???)
平均AUC = ???
AUC標準差 = ???
print(每摺AUC.round(3), round(平均AUC, 3), round(AUC標準差, 3))

In [ ]:
檢查("7-2")   # ◀ 執行這一格，看看任務 7-2 有沒有過關

### 🎯 任務 7-3　OOF AUC 的 bootstrap 區間

用 `cross_val_predict(pipe, X_train, y_train, cv=5, method='predict_proba')[:, 1]` 取得 `OOF機率`，算出 `OOF_AUC`；再用種子 42 的 `rng` 對 `y_train` 與 `OOF機率` 抽後放回 500 次，算出 `AUC下界`、`AUC上界`（2.5 與 97.5 百分位）。final test 仍不可使用。

In [ ]:
# 🎯 任務 7-3　OOF AUC 的 bootstrap 區間（請保留這一行）
OOF機率 = cross_val_predict(pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
OOF_AUC = ???
rng = np.random.default_rng(42)
索引 = np.arange(len(y_train)); yv = y_train.values
AUC們 = []
for _ in range(500):
    s = rng.choice(索引, len(索引), replace=True)
    if np.unique(yv[s]).size < 2: continue
    AUC們.append(roc_auc_score(yv[s], OOF機率[s]))
AUC下界 = ???
AUC上界 = ???
print(round(OOF_AUC, 3), round(AUC下界, 3), round(AUC上界, 3))

In [ ]:
檢查("7-3")   # ◀ 執行這一格，看看任務 7-3 有沒有過關

### 🎯 任務 7-4　一棵樹 vs 一座森林

建立 `tree`（`DecisionTreeClassifier(random_state=42)`）與 `rf`（`RandomForestClassifier(n_estimators=300, random_state=42)`），只在 `X_train, y_train` 做相同 5 摺 CV，算 `樹CV平均`、`森林CV平均`。fit 樹後算 `樹訓練AUC`，以訓練 AUC − CV 平均 > 0.2 判斷 `樹在背考古題`。依 CV 較高者設定 `選定模型` 與 `最終模型`；最後才 fit `最終模型`，開封 final test 一次，將 AUC 存成 `最終測試AUC`。

In [ ]:
# 🎯 任務 7-4　一棵樹 vs 一座森林（請保留這一行）
tree = Pipeline([("preprocess", preprocess), ("model", DecisionTreeClassifier(random_state=42))])
rf = Pipeline([("preprocess", preprocess), ("model", RandomForestClassifier(n_estimators=300, random_state=42))])
樹CV = cross_val_score(tree, X_train, y_train, cv=5, scoring="roc_auc")
森林CV = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
樹CV平均 = 樹CV.mean()
森林CV平均 = 森林CV.mean()
tree.fit(X_train, y_train)
樹訓練AUC = roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1])
樹在背考古題 = (樹訓練AUC - 樹CV平均) > 0.2
選定模型 = ???
最終模型 = ???
最終模型.fit(X_train, y_train)
最終測試AUC = ???
print(round(樹CV平均, 3), round(森林CV平均, 3), 選定模型, round(最終測試AUC, 3), 樹在背考古題)

In [ ]:
檢查("7-4")   # ◀ 執行這一格，看看任務 7-4 有沒有過關

### 🎯 任務 7-5　特徵重要性

從已選定並完成 final test 的 `最終模型` Pipeline 取出 `model.feature_importances_` 與前處理後的欄名，做成 Series `重要性`（由大到小排序），取出 `最重要` 的特徵名稱，並畫成橫條圖（標題含「重要」）。這一步只解讀已選定模型，不再回頭比較 test 分數。

In [ ]:
# 🎯 任務 7-5　特徵重要性（請保留這一行）
特徵名稱 = 最終模型.named_steps["preprocess"].get_feature_names_out()
重要性 = pd.Series(最終模型.named_steps["model"].feature_importances_, index=特徵名稱).sort_values(ascending=???)
最重要 = 重要性.index[0]
重要性.plot(kind="barh", title="隨機森林特徵重要性"); plt.gca().invert_yaxis(); plt.show()
print(重要性.round(3)); print("最重要：", 最重要)

In [ ]:
檢查("7-5")   # ◀ 執行這一格，看看任務 7-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把 `cv=5` 改成 `cv=10`，平均 AUC 和標準差怎麼變？
2. 隨機森林的 `n_estimators` 從 10 → 50 → 300，只用訓練內 CV 比較分數與時間；不要反覆查看 final test。

---
## 🔑 通關密語
　你已經知道怎麼讓分數「可信」，也有了一座森林。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🧲 L08 正規化與調參** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.0.0-rc.1/notebooks/L08_regularization.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.1/